<img src="../assets/logo-banner.png" alt="radar-datatree — Cloud-native, time-aware weather radar datasets" width="800">

# radar datatree
---

`Radar datatree` is a [FAIR](https://www.go-fair.org/fair-principles/) and cloud-native framework that turns fragmented NEXRAD Level II archives — millions of standalone binary files — into hierarchical, time-indexed, analysis-ready datasets queryable directly from object storage.

## Why a DataTree?

Weather radar observations come in two shapes: individual 360-degree sweeps at a fixed antenna elevation angle, and collections of such sweeps stacked into three-dimensional volumes (a *Volume Coverage Pattern*, or VCP). The radar repeats a VCP every 4 to 10 minutes depending on the weather. Sweeps from different VCPs can have different shapes or dimensions, which makes `xarray.DataTree` a perfect data model to represent and store radar data.

- **Hierarchical** — every VCP × sweep combination is a node you can navigate (`dt["VCP-12/sweep_0"]`), not a separate file you have to download and parse.
- **Time-indexed** — every sweep stacks all of its scans along a `vcp_time` dimension, so `.sel(vcp_time=...)` replaces the file-iteration loop.
- **Lazy and cloud-native** — opening the full archive fetches metadata only (~MB). Variables stream from object storage on demand when you reduce or plot them.

For the architecture and the scaling benchmarks, see [Ladino-Rincón & Nesbitt (2025)](https://doi.org/10.48550/arXiv.2510.24943).

## radar datatree in practice

Let's see how the radar datatree looks in practice.

In [1]:
import icechunk

# Public radar-datatree archives on the AWS Open Data Registry.
# Create a connection and session to the icechunkv3 - Zarrv3 repo
storage = icechunk.s3_storage(
    bucket="nexrad-arco",
    prefix="KLOT",
    region="us-east-1",
    anonymous=True,
)
session = icechunk.Repository.open(storage).readonly_session("main")

We can use [`xarray.open_datatree`](https://docs.xarray.dev/en/stable/generated/xarray.open_datatree.html) to explore the radar archive.

In [2]:
import xarray as xr
import xradar  # noqa: F401  — registers the .xradar accessor

dt = xr.open_datatree(session.store, engine="rustytree", chunks=None)
dt

<xarray.DataTree>
Group: /
├── Group: /VCP-112
│   │   Dimensions:        (vcp_time: 108)
│   │   Coordinates:
│   │     * vcp_time       (vcp_time) datetime64[ns] 864B 2020-11-05T13:16:18.795000 ...
│   │       altitude       int64 8B ...
│   │       latitude       float64 8B ...
│   │       longitude      float64 8B ...
│   │   Data variables:
│   │       volume_number  (vcp_time) float64 864B ...
│   │   Attributes:
│   │       Conventions:  Cf/Radial instrument_parameters radar_parameters
│   │       attribution:  NOAA NEXRAD Level 2 data processed by Atmoscale from NOAA O...
│   │       dataset_id:   nexrad-arco-klot
│   │       institution:  NOAA National Weather Service
│   │       source:       WSR-88D S-band weather radar
│   │       time_domain:  2026-05-08 to Present
│   │       title:        NEXRAD ARCO - KLOT
│   │       version:      2.1
│   ├── Group: /VCP-112/sweep_0
│   │       Dimensions:              (vcp_time: 108, azimuth: 720, range: 1832)
│   │       Coordinates:
│   │         * azimuth              (azimuth) float64 6kB 0.25 0.75 1.25 ... 359.2 359.8
│   │           elevation            (azimuth) float64 6kB ...
│   │           time                 (vcp_time, azimuth) datetime64[ns] 622kB ...
│   │         * range                (range) float32 7kB 2.125e+03 2.375e+03 ... 4.599e+05
│   │       Data variables:
│   │           CCORH                (vcp_time, azimuth, range) float32 570MB ...
│   │           DBZH                 (vcp_time, azimuth, range) float32 570MB ...
│   │           PHIDP                (vcp_time, azimuth, range) float32 570MB ...
│   │           RHOHV                (vcp_time, azimuth, range) float32 570MB ...
│   │           ZDR                  (vcp_time, azimuth, range) float32 570MB ...
│   │           ray_elevation_angle  (vcp_time, azimuth) float64 622kB ...
│   │           sweep_fixed_angle    (vcp_time) float32 432B ...
│   │           sweep_number         (vcp_time) float32 432B ...
│   ├── Group: /VCP-112/sweep_1
│   │       Dimensions:              (vcp_time: 108, azimuth: 720, range: 1192)
│   │       Coordinates:
│   │         * azimuth              (azimuth) float64 6kB 0.25 0.75 1.25 ... 359.2 359.8
│   │           elevation            (azimuth) float64 6kB ...
│   │           time                 (vcp_time, azimuth) datetime64[ns] 622kB ...
│   │         * range                (range) float32 5kB 2.125e+03 2.375e+03 ... 2.999e+05
│   │       Data variables:
│   │           DBZH                 (vcp_time, azimuth, range) float32 371MB ...
│   │           VRADH                (vcp_time, azimuth, range) float32 371MB ...
│   │           WRADH                (vcp_time, azimuth, range) float32 371MB ...
│   │           ray_elevation_angle  (vcp_time, azimuth) float64 622kB ...
│   │           sweep_fixed_angle    (vcp_time) float32 432B ...
│   │           sweep_number         (vcp_time) float32 432B ...
│   ├── Group: /VCP-112/sweep_10
│   │       Dimensions:              (vcp_time: 108, azimuth: 360, range: 1336)
│   │       Coordinates:
│   │         * azimuth              (azimuth) float64 3kB 0.5 1.5 2.5 ... 357.5 358.5 359.5
│   │           elevation            (azimuth) float64 3kB ...
│   │           time                 (vcp_time, azimuth) datetime64[ns] 311kB ...
│   │         * range                (range) float32 5kB 2.125e+03 2.375e+03 ... 3.359e+05
│   │       Data variables:
│   │           CCORH                (vcp_time, azimuth, range) float32 208MB ...
│   │           DBZH                 (vcp_time, azimuth, range) float32 208MB ...
│   │           PHIDP                (vcp_time, azimuth, range) float32 208MB ...
│   │           RHOHV                (vcp_time, azimuth, range) float32 208MB ...
│   │           VRADH                (vcp_time, azimuth, range) float32 208MB ...
│   │           WRADH                (vcp_time, azimuth, range) float32 208MB ...
│   │           ZDR                  (vcp_time, azimuth, range) float32 208MB ...
│   │           ray

Opened with `engine="rustytree"` — a Rust-backed xarray DataTree backend recommended for radar-datatree archives. Drop-in replacement for the standard `engine="zarr"`, ~10× faster on icechunk repos served from object storage. See [`rustytree-xarray` on PyPI](https://pypi.org/project/rustytree-xarray/).

We can access any of these VCPs using a file-path syntax — for example, `dt["VCP-212/sweep_0"]` — which returns an `xarray.Dataset`. Before slicing anything though, let's see how big the full archive is.

In [3]:
print(f"datatree size: {dt.nbytes / 1024**4:.2f} TB")

datatree size: 92.38 TB


That's almost 100 TB of radar data spanning January 2020 to May 2026 — too much to hold in a single session. Fortunately, we don't have to: we can open only the sweeps or VCPs we care about.

## Opening only what you need

`xarray` and `zarr` let us inspect a dataset's structure by pulling metadata only — no data is read until we ask for it. The `group=` parameter slices the tree at open time. Below, `"*/sweep_0"` returns the lowest-elevation cut (`sweep_0`) from every VCP; the parent VCP nodes are auto-included as ancestors.

In [4]:
dt_sweep0 = xr.open_datatree(session.store, engine="rustytree", group="*/sweep_0")

In [5]:
dt_sweep0

<xarray.DataTree>
Group: /
├── Group: /VCP-112
│   │   Dimensions:        (vcp_time: 108)
│   │   Coordinates:
│   │     * vcp_time       (vcp_time) datetime64[ns] 864B 2020-11-05T13:16:18.795000 ...
│   │       altitude       int64 8B ...
│   │       latitude       float64 8B ...
│   │       longitude      float64 8B ...
│   │   Data variables:
│   │       volume_number  (vcp_time) float64 864B ...
│   │   Attributes:
│   │       Conventions:  Cf/Radial instrument_parameters radar_parameters
│   │       attribution:  NOAA NEXRAD Level 2 data processed by Atmoscale from NOAA O...
│   │       dataset_id:   nexrad-arco-klot
│   │       institution:  NOAA National Weather Service
│   │       source:       WSR-88D S-band weather radar
│   │       time_domain:  2026-05-08 to Present
│   │       title:        NEXRAD ARCO - KLOT
│   │       version:      2.1
│   └── Group: /VCP-112/sweep_0
│           Dimensions:              (vcp_time: 108, azimuth: 720, range: 1832)
│           Coordinates:
│             * azimuth              (azimuth) float64 6kB 0.25 0.75 1.25 ... 359.2 359.8
│               elevation            (azimuth) float64 6kB ...
│               time                 (vcp_time, azimuth) datetime64[ns] 622kB ...
│             * range                (range) float32 7kB 2.125e+03 2.375e+03 ... 4.599e+05
│           Data variables:
│               CCORH                (vcp_time, azimuth, range) float32 570MB ...
│               DBZH                 (vcp_time, azimuth, range) float32 570MB ...
│               PHIDP                (vcp_time, azimuth, range) float32 570MB ...
│               RHOHV                (vcp_time, azimuth, range) float32 570MB ...
│               ZDR                  (vcp_time, azimuth, range) float32 570MB ...
│               ray_elevation_angle  (vcp_time, azimuth) float64 622kB ...
│               sweep_fixed_angle    (vcp_time) float32 432B ...
│               sweep_number         (vcp_time) float32 432B ...
├── Group: /VCP-12
│   │   Dimensions:        (vcp_time: 2573)
│   │   Coordinates:
│   │     * vcp_time       (vcp_time) datetime64[ns] 21kB 2020-02-10T08:33:43.155000 ...
│   │       altitude       int64 8B ...
│   │       latitude       float64 8B ...
│   │       longitude      float64 8B ...
│   │   Data variables:
│   │       volume_number  (vcp_time) float64 21kB ...
│   │   Attributes:
│   │       Conventions:  Cf/Radial instrument_parameters radar_parameters
│   │       attribution:  NOAA NEXRAD Level 2 data processed by Atmoscale from NOAA O...
│   │       dataset_id:   nexrad-arco-klot
│   │       institution:  NOAA National Weather Service
│   │       source:       WSR-88D S-band weather radar
│   │       time_domain:  2026-05-08 to Present
│   │       title:        NEXRAD ARCO - KLOT
│   │       version:      2.1
│   └── Group: /VCP-12/sweep_0
│           Dimensions:              (vcp_time: 2573, azimuth: 720, range: 1832)
│           Coordinates:
│             * azimuth              (azimuth) float64 6kB 0.25 0.75 1.25 ... 359.2 359.8
│               elevation            (azimuth) float64 6kB ...
│               time                 (vcp_time, azimuth) datetime64[ns] 15MB ...
│             * range                (range) float32 7kB 2.125e+03 2.375e+03 ... 4.599e+05
│           Data variables:
│               CCORH                (vcp_time, azimuth, range) float32 14GB ...
│               DBZH                 (vcp_time, azimuth, range) float32 14GB ...
│               PHIDP                (vcp_time, azimuth, range) float32 14GB ...
│               RHOHV                (vcp_time, azimuth, range) float32 14GB ...
│               ZDR                  (vcp_time, azimuth, range) float32 14GB ...
│               ray_elevation_angle  (vcp_time, azimuth) float64 15MB ...
│               sweep_fixed_angle    (vcp_time) float32 10kB ...
│               sweep_number         (vcp_time) float32 10kB ...
├── Group: /VCP-212
│   │   Dimensions:        (vcp_time: 124562)
│   │   Coordinates:
│   │  

Similarly, we can target a single VCP and sweep:

In [6]:
# Open only VCP-212 / sweep_0 (severe-weather rapid-scan mode, lowest elevation)
dt_212 = xr.open_datatree(
    session.store,
    engine="rustytree",
    group="VCP-212/sweep_0",
)

In [7]:
dt_212

<xarray.DataTree>
Group: /
    Dimensions:              (vcp_time: 124562, azimuth: 720, range: 1832)
    Coordinates:
      * azimuth              (azimuth) float64 6kB 0.25 0.75 1.25 ... 359.2 359.8
        elevation            (azimuth) float64 6kB ...
        time                 (vcp_time, azimuth) datetime64[ns] 717MB ...
      * range                (range) float32 7kB 2.125e+03 2.375e+03 ... 4.599e+05
    Dimensions without coordinates: vcp_time
    Data variables:
        CCORH                (vcp_time, azimuth, range) float32 657GB ...
        DBZH                 (vcp_time, azimuth, range) float32 657GB ...
        PHIDP                (vcp_time, azimuth, range) float32 657GB ...
        RHOHV                (vcp_time, azimuth, range) float32 657GB ...
        ZDR                  (vcp_time, azimuth, range) float32 657GB ...
        ray_elevation_angle  (vcp_time, azimuth) float64 717MB ...
        sweep_fixed_angle    (vcp_time) float32 498kB ...
        sweep_number         (vcp_time) float32 498kB ...

## Plot a polarimetric snapshot

Now that the data is **analysis-ready and cloud-optimized**, plotting is the easy part. Let's grab a single timestamp from `VCP-34/sweep_0` (clear-air mode, ~0.5° elevation) during the December 2025 KLOT winter storm and georeference it so the polar (azimuth, range) gates land on Cartesian (x, y) axes.

In [8]:
scan = (
    dt["VCP-34/sweep_0"]
    .to_dataset(inherit="all_coords")
    .sel(vcp_time="2025-12-13 13:10", method="nearest")
    .xradar.georeference()
)

In [9]:
scan

<xarray.Dataset> Size: 58MB
Dimensions:              (azimuth: 720, range: 1832)
Coordinates:
  * azimuth              (azimuth) float64 6kB 0.25 0.75 1.25 ... 359.2 359.8
    elevation            (azimuth) float64 6kB 0.5 0.5 0.5 0.5 ... 0.5 0.5 0.5
    time                 (azimuth) datetime64[ns] 6kB ...
  * range                (range) float32 7kB 2.125e+03 2.375e+03 ... 4.599e+05
    x                    (azimuth, range) float64 11MB 9.271 ... -2.004e+03
    y                    (azimuth, range) float64 11MB 2.125e+03 ... 4.592e+05
    z                    (azimuth, range) float64 11MB 249.8 252.1 ... 1.668e+04
    altitude             float64 8B 231.0
    latitude             float64 8B 41.6
    longitude            float64 8B -88.08
    vcp_time             datetime64[ns] 8B 2025-12-13T13:12:30.487999916
    crs_wkt              int64 8B 0
Data variables:
    CCORH                (azimuth, range) float32 5MB ...
    DBZH                 (azimuth, range) float32 5MB ...
    PHIDP                (azimuth, range) float32 5MB ...
    RHOHV                (azimuth, range) float32 5MB ...
    ZDR                  (azimuth, range) float32 5MB ...
    ray_elevation_angle  (azimuth) float64 6kB ...
    sweep_fixed_angle    float32 4B ...
    sweep_number         float32 4B ...

In [ ]:
import cmweather  # noqa: F401  — registers ChaseSpectral colormap
import matplotlib.pyplot as plt

# Rescale x/y from meters to kilometers for legible tick marks.
scan_km = scan.assign_coords(x=scan.x / 1000, y=scan.y / 1000)
scan_km.x.attrs["units"] = "km"
scan_km.y.attrs["units"] = "km"

fig, ax = plt.subplots(figsize=(7, 6))
scan_km.DBZH.plot(x="x", y="y", cmap="ChaseSpectral", vmin=-10, vmax=70, ax=ax)
ax.set_title(f"DBZH — {str(scan.vcp_time.values)[:19]} UTC")
ax.set_xlabel("x [km]")
ax.set_ylabel("y [km]")
ax.set_aspect("equal")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Rescale x/y from meters to kilometers for legible tick marks.
scan_km = scan.assign_coords(x=scan.x / 1000, y=scan.y / 1000)
scan_km.x.attrs["units"] = "km"
scan_km.y.attrs["units"] = "km"

fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
panels = [
    ("DBZH", "ChaseSpectral", -10, 70, "Reflectivity [dBZ]"),
    ("ZDR", "HomeyerRainbow", -2, 6, "Differential Reflectivity [dB]"),
    ("RHOHV", "Carbone11", 0.7, 1.0, "Cross-Correlation Coefficient"),
    ("PHIDP", "PD17", 0, 180, "Differential Phase [deg]"),
]
for ax, (var, cmap, vmin, vmax, label) in zip(axes.flat, panels, strict=True):
    scan_km[var].plot(
        ax=ax,
        x="x",
        y="y",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        cbar_kwargs={"label": label, "shrink": 0.8},
    )
    ax.set_title(var)
    ax.set_xlabel("x [km]")
    ax.set_ylabel("y [km]")
    ax.set_aspect("equal")
fig.suptitle(
    f"KLOT polarimetric snapshot — {str(scan.vcp_time.values)[:19]} UTC",
    fontsize=12,
)
plt.tight_layout()
plt.show()

## Where to next

- **[Notebook 2 — Reproduce Ryzhkov et al. (2016)](2.QVP-Workflow-Comparison)** — the laptop-runnable QVP comparison + numerical-equivalence assertion across both paths.
- **[Notebook 3 — QPE scaling benchmark](3.QPE-Scaling-Benchmark)** — Marshall–Palmer rainfall accumulation, 1-day live + cluster-recommended templates for 7d / 30d / 6mo windows.
- **[radar-datatree on GitHub](https://github.com/AtmoScale/radar-datatree)** — the framework's source, full documentation, and additional tutorials.
- **[AWS Open Data Registry — nexrad-arco](https://registry.opendata.aws/nexrad-arco/)** — dataset metadata, terms of use, and updates as more radars are published.
- **[rustytree-xarray](https://github.com/aladinor/rustytree)** — the Rust DataTree backend.

---

*Cite this work:* Ladino-Rincón, A., & Nesbitt, S. W. (2025). *Radar DataTree: A FAIR and Cloud-Native Framework for Scalable Weather Radar Archives.* arXiv:2510.24943. [doi:10.48550/arXiv.2510.24943](https://doi.org/10.48550/arXiv.2510.24943)